1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



# 🟪 Grupo D — Variables categóricas (dummies por canal y día)

**Variables:**
- `data_channel_is_*`
- `weekday_is_*`
- `is_weekend`

**👉 Aquí analizamos:**
- qué canales publican contenido más viral  
- qué días generan más shares  


🟪 Tabla de descripción de variables — Grupo D (Canales y Días)

| Variable                       | Tipo  | Descripción                                                                 | Interpretación / Aporte al modelo                                                                                           |
|--------------------------------|-------|-----------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------|
| data_channel_is_world          | Dummy | La noticia pertenece al canal **World**.                                   | Este canal suele tener alto volumen; su efecto en shares puede ser moderado, aporta segmentación por temática global.        |
| data_channel_is_tech           | Dummy | La noticia pertenece al canal **Tech**.                                    | Puede atraer audiencias especializadas; aporta señal si el contenido técnico tiende a ser más o menos viral.                |
| data_channel_is_entertainment  | Dummy | La noticia pertenece al canal **Entertainment**.                           | Suele tener mayores shares; aporta fuerte señal de viralidad por contenido ligero y atractivo.                              |
| data_channel_is_bus            | Dummy | La noticia pertenece al canal **Business**.                                | Tiende a generar menos shares; aporta identificación de contenido serio o técnico con baja viralidad.                       |
| data_channel_is_socmed         | Dummy | La noticia pertenece al canal **Social Media**.                            | Es uno de los más virales; aporta una señal muy fuerte para predecir mayores shares.                                         |
| data_channel_is_lifestyle      | Dummy | La noticia pertenece al canal **Lifestyle**.                               | Puede atraer lectores casuales; aporta segmentación de nichos con patrones de viralidad variables.                          |
|                                |       |                                                                             |                                                                                                                              |
| weekday_is_monday              | Dummy | Publicado en **lunes**.                                                    | Puede tener menor rendimiento; aporta señal de menor actividad de usuarios tras el fin de semana.                           |
| weekday_is_tuesday             | Dummy | Publicado en **martes**.                                                   | Día con tráfico más estable; aporta una referencia útil de comportamiento “normal”.                                          |
| weekday_is_wednesday           | Dummy | Publicado en **miércoles**.                                                | Suele tener alto volumen; buen punto de comparación para días con más engagement.                                           |
| weekday_is_thursday            | Dummy | Publicado en **jueves**.                                                   | Frecuentemente uno de los días más virales; aporta señal positiva por mayor actividad antes del fin de semana.              |
| weekday_is_friday              | Dummy | Publicado en **viernes**.                                                  | Puede generar alta viralidad; aporta señal de contenido compartido previo al descanso.                                       |
| weekday_is_saturday            | Dummy | Publicado en **sábado**.                                                   | Menor actividad; aporta señal negativa típica de bajo engagement.                                                            |
| weekday_is_sunday              | Dummy | Publicado en **domingo**.                                                  | Similar al sábado pero con ligera recuperación; aporta referencia para días de descanso.                                     |
|                                |       |                                                                             |                                                                                                                              |
| is_weekend                     | Dummy | Publicado en fin de semana (sábado o domingo).                             | Generalmente se asocia a menos shares; aporta señal clara para diferenciar periodos de baja actividad.                      |


✅ Estadísticas Descriptivas del Grupo D (Canales y Días)

In [ ]:
# =====================================================================
# 🟪 Estadísticas Descriptivas — Grupo D (Canales y Días)
# =====================================================================

import pandas as pd
import numpy as np

# ----------------------------
# 1. Identificar columnas
# ----------------------------
channel_cols = [col for col in df.columns if col.startswith("data_channel_is_")]
weekday_cols = [col for col in df.columns if col.startswith("weekday_is_")]
weekend_col = "is_weekend"


# =====================================================================
# 2. Estadísticas descriptivas para CANALES
# =====================================================================

channel_stats = pd.DataFrame({
    "frecuencia": df[channel_cols].sum(),
    "porcentaje": df[channel_cols].mean() * 100,
})

# Agregar estadísticas de shares
avg_shares_channel = (
    df[channel_cols + ["shares"]]
    .melt(id_vars="shares", var_name="channel", value_name="is_channel")
    .query("is_channel == 1")
    .groupby("channel")["shares"]
    .agg(["mean", "median", "std"])
)

channel_stats = channel_stats.join(avg_shares_channel)
channel_stats = channel_stats.sort_values("frecuencia", ascending=False)

print("📌 Estadísticas descriptivas — CANALES")
display(channel_stats)


# =====================================================================
# 3. Estadísticas descriptivas para DÍAS DE LA SEMANA
# =====================================================================

weekday_stats = pd.DataFrame({
    "frecuencia": df[weekday_cols].sum(),
    "porcentaje": df[weekday_cols].mean() * 100,
})

# Agregar estadísticas de shares por día
avg_shares_weekday = (
    df[weekday_cols + ["shares"]]
    .melt(id_vars="shares", var_name="weekday", value_name="is_day")
    .query("is_day == 1")
    .groupby("weekday")["shares"]
    .agg(["mean", "median", "std"])
)

weekday_stats = weekday_stats.join(avg_shares_weekday)
weekday_stats = weekday_stats.sort_values("frecuencia", ascending=False)

print("\n📌 Estadísticas descriptivas — DÍAS DE LA SEMANA")
display(weekday_stats)


# =====================================================================
# 4. Estadísticas descriptivas — Fin de semana (is_weekend)
# =====================================================================

weekend_stats = df.groupby(weekend_col)["shares"].agg(
    frecuencia = "count",
    mean = "mean",
    median = "median",
    std = "std"
)

# Renombrar índices para claridad
weekend_stats.index = ["Entre semana (0)", "Fin de semana (1)"]

print("\n📌 Estadísticas descriptivas — FIN DE SEMANA vs ENTRE SEMANA")
display(weekend_stats)



✅ 1. Frecuencia de publicaciones por canal

In [ ]:
# Frecuencia de publicaciones por canal
channel_cols = [col for col in df.columns if col.startswith("data_channel_is_")]

channel_counts = df[channel_cols].sum().sort_values(ascending=False)
display(channel_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x=channel_counts.index, y=channel_counts.values)
plt.title("Frecuencia de publicaciones por canal")
plt.xticks(rotation=45)
plt.ylabel("Número de publicaciones")
plt.show()


✅ 2. Shares promedio por canal

In [ ]:
# Shares promedio por canal
avg_shares_by_channel = (
    df[channel_cols + ["shares"]]
    .melt(id_vars="shares", var_name="channel", value_name="is_channel")
    .query("is_channel == 1")
    .groupby("channel")["shares"]
    .mean()
    .sort_values(ascending=False)
)

display(avg_shares_by_channel)

plt.figure(figsize=(10, 5))
sns.barplot(x=avg_shares_by_channel.index, y=avg_shares_by_channel.values)
plt.title("Shares promedio por canal")
plt.xticks(rotation=45)
plt.ylabel("Promedio de shares")
plt.show()


✅ 3. Frecuencia de publicaciones por día

In [ ]:
# Frecuencia de publicaciones por día
weekday_cols = [col for col in df.columns if col.startswith("weekday_is_")]

weekday_counts = df[weekday_cols].sum().sort_values(ascending=False)
display(weekday_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x=weekday_counts.index, y=weekday_counts.values)
plt.title("Frecuencia de publicaciones por día")
plt.xticks(rotation=45)
plt.ylabel("Número de publicaciones")
plt.show()


✅ 4. Shares promedio por día

In [ ]:
# Shares promedio por día
avg_shares_by_weekday = (
    df[weekday_cols + ["shares"]]
    .melt(id_vars="shares", var_name="weekday", value_name="is_day")
    .query("is_day == 1")
    .groupby("weekday")["shares"]
    .mean()
    .sort_values(ascending=False)
)

display(avg_shares_by_weekday)

plt.figure(figsize=(10, 5))
sns.barplot(x=avg_shares_by_weekday.index, y=avg_shares_by_weekday.values)
plt.title("Shares promedio por día")
plt.xticks(rotation=45)
plt.ylabel("Promedio de shares")
plt.show()


✅ 5. Comparación fin de semana vs semana

In [ ]:
# Comparación fin de semana vs semana
weekend_col = "is_weekend"

weekend_comparison = df.groupby(weekend_col)["shares"].mean()
weekend_comparison.index = ["Entre semana (0)", "Fin de semana (1)"]

display(weekend_comparison)

plt.figure(figsize=(7, 5))
sns.barplot(x=weekend_comparison.index, y=weekend_comparison.values)
plt.title("Shares: Semana vs Fin de semana")
plt.ylabel("Promedio de shares")
plt.show()


✅ 6. Heatmap Canal × Día (promedio de shares)

In [ ]:
# Heatmap Canal × Día (promedio de shares)
pivot_heatmap = (
    df[channel_cols + weekday_cols + ["shares"]]
    .melt(id_vars=channel_cols + ["shares"], var_name="weekday", value_name="is_day")
    .query("is_day == 1")
    .melt(id_vars=["weekday", "shares"], var_name="channel", value_name="is_channel")
    .query("is_channel == 1")
    .groupby(["channel", "weekday"])["shares"]
    .mean()
    .unstack()
)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_heatmap, cmap="viridis", annot=False)
plt.title("Heatmap de shares promedio: Canal vs Día")
plt.show()


🟪 7. Distribución de shares por canal y por día

🔹 7.1 Boxplots de shares por canal (escala log)

In [ ]:
# =====================================================================
# 7. Distribución de shares por canal y día (boxplots)
# =====================================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Asegurar listas de columnas (por si no están en este bloque)
channel_cols = [col for col in df.columns if col.startswith("data_channel_is_")]
weekday_cols = [col for col in df.columns if col.startswith("weekday_is_")]
weekend_col = "is_weekend"

# ---------------------------------------------------------
# 7.1 Boxplot de shares por canal (usando escala log10)
# ---------------------------------------------------------
channel_long = (
    df[channel_cols + ["shares"]]
    .melt(id_vars="shares", var_name="channel", value_name="is_channel")
    .query("is_channel == 1")
    .copy()
)

channel_long["log_shares"] = np.log10(channel_long["shares"] + 1)

plt.figure(figsize=(10, 6))
sns.boxplot(data=channel_long, x="channel", y="log_shares")
plt.title("Distribución de shares (log10) por canal")
plt.xlabel("Canal")
plt.ylabel("log10(shares + 1)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


7.2 Boxplots de shares por día de la semana (escala log)

In [ ]:
# ---------------------------------------------------------
# 7.2 Boxplot de shares por día de la semana (log10)
# ---------------------------------------------------------
weekday_long = (
    df[weekday_cols + ["shares"]]
    .melt(id_vars="shares", var_name="weekday", value_name="is_day")
    .query("is_day == 1")
    .copy()
)

weekday_long["log_shares"] = np.log10(weekday_long["shares"] + 1)

plt.figure(figsize=(10, 6))
sns.boxplot(data=weekday_long, x="weekday", y="log_shares")
plt.title("Distribución de shares (log10) por día de la semana")
plt.xlabel("Día de la semana")
plt.ylabel("log10(shares + 1)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7. Distribución de shares por canal y día

Al analizar la distribución de *shares* usando boxplots en escala logarítmica, observamos que:

- La distribución es altamente asimétrica, con colas largas hacia la derecha (pocas noticias extremadamente virales).
- Algunos canales como `data_channel_is_socmed` y `data_channel_is_lifestyle` muestran medianas más altas y presencia de outliers fuertes.
- En los días de la semana, sábado y domingo presentan medianas de *shares* más elevadas y mayor dispersión, confirmando que el fin de semana concentra publicaciones más virales.

Esto confirma que no basta con la media: la distribución es muy sesgada, y los outliers juegan un papel importante en la viralidad.


🟪 8. Pruebas estadísticas: ¿las diferencias son significativas?

🔹 8.1 ANOVA: shares entre canales

In [ ]:
# =====================================================================
# 8. Pruebas estadísticas: ANOVA y t-test
# =====================================================================
from scipy.stats import f_oneway, ttest_ind

# -------------------------------
# 8.1 ANOVA: shares ~ canal
# -------------------------------
groups_channels = []
labels_channels = []

for col in channel_cols:
    mask = df[col] == 1
    groups_channels.append(df.loc[mask, "shares"].values)
    labels_channels.append(col)

f_stat_channel, p_value_channel = f_oneway(*groups_channels)

print("ANOVA - Comparación de shares entre canales")
print(f"F = {f_stat_channel:.4f}, p-valor = {p_value_channel:.4e}")


🔹 8.2 ANOVA: shares entre días de la semana

In [ ]:
# -------------------------------
# 8.2 ANOVA: shares ~ día de la semana
# -------------------------------
groups_weekdays = []
labels_weekdays = []

for col in weekday_cols:
    mask = df[col] == 1
    groups_weekdays.append(df.loc[mask, "shares"].values)
    labels_weekdays.append(col)

f_stat_weekday, p_value_weekday = f_oneway(*groups_weekdays)

print("ANOVA - Comparación de shares entre días de la semana")
print(f"F = {f_stat_weekday:.4f}, p-valor = {p_value_weekday:.4e}")


🔹 8.3 t-test: entre semana vs fin de semana

In [ ]:
# -------------------------------
# 8.3 t-test: entre semana vs fin de semana
# -------------------------------
shares_week = df.loc[df["is_weekend"] == 0, "shares"]
shares_weekend = df.loc[df["is_weekend"] == 1, "shares"]

t_stat, p_value_t = ttest_ind(shares_week, shares_weekend, equal_var=False)

print("t-test - Shares: entre semana vs fin de semana")
print(f"t = {t_stat:.4f}, p-valor = {p_value_t:.4e}")


### 8. Pruebas estadísticas (ANOVA y t-test)

Para validar si las diferencias observadas son reales y no producto del azar, se aplicaron pruebas estadísticas:

- **ANOVA por canal (`data_channel_is_*`)**: el p-valor resultó extremadamente pequeño (p ≪ 0.05), lo que indica que existen diferencias significativas en el número de *shares* entre los distintos canales.
- **ANOVA por día de la semana (`weekday_is_*`)**: nuevamente, el p-valor fue muy bajo, confirmando diferencias significativas entre los días de publicación.
- **t-test entre semana vs fin de semana (`is_weekend`)**: el p-valor también fue menor a 0.05, lo que respalda que las publicaciones de fin de semana tienen un comportamiento de *shares* estadísticamente distinto (y mayor) que las de entre semana.

En resumen, canal, día y la condición de fin de semana **sí aportan información relevante** y diferenciada para explicar la viralidad.


🟪 9. Top combinaciones Canal × Día (ranking de viralidad)

In [ ]:
# =====================================================================
# 9. Ranking de combinaciones Canal × Día
# =====================================================================

# Construir tabla canal × día con promedio de shares
combo_df = (
    df[channel_cols + weekday_cols + ["shares"]]
    .melt(id_vars=channel_cols + ["shares"], var_name="weekday", value_name="is_day")
    .query("is_day == 1")
    .melt(id_vars=["weekday", "shares"], var_name="channel", value_name="is_channel")
    .query("is_channel == 1")
    .groupby(["channel", "weekday"])
    .agg(
        mean_shares=("shares", "mean"),
        median_shares=("shares", "median"),
        freq=("shares", "count")
    )
    .reset_index()
    .sort_values("mean_shares", ascending=False)
)

print("📌 Top 10 combinaciones Canal × Día más virales (por promedio de shares)")
display(combo_df.head(10))


### Interpretación del Top 10 Canal × Día

Esta tabla muestra las combinaciones específicas de **canal + día** que generan los niveles más altos de viralidad (medidos por promedio de shares). Los principales hallazgos son:

- **El fin de semana concentra la mayor viralidad**: domingo, sábado y viernes aparecen en casi todas las combinaciones del top 10.
- **Los canales Social Media, Lifestyle y Tech dominan la parte alta de la tabla**, especialmente cuando publican en domingo.
- **Lifestyle y Tech funcionan especialmente bien los fines de semana**, indicando un comportamiento de consumo relajado y mayor disposición a compartir contenido.
- **Business sorprende los sábados**, mostrando que ciertos temas de negocios pueden tener mejor recepción fuera del horario laboral.
- Las medianas elevadas (2000–2400 shares) confirman que estas combinaciones no sólo se explican por outliers, sino por un patrón real de mayor viralidad.
- En conjunto, estos resultados refuerzan que la interacción Canal × Día es clave para entender cuándo un artículo alcanza su máximo potencial de shares.

Este análisis es muy valioso para optimizar estrategias de publicación y también para el modelo predictivo, ya que demuestra que canal y día aportan una señal fuerte y diferenciada.


🟪 10. Correlación entre dummies de canales y días

In [ ]:
# =====================================================================
# 10. Correlación entre variables dummies del Grupo D
# =====================================================================

groupD_cols = channel_cols + weekday_cols + [weekend_col]

corr_matrix = df[groupD_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0)
plt.title("Matriz de correlación — Grupo D (Canales y Días)")
plt.tight_layout()
plt.show()


### 10. Correlación entre variables categóricas (dummies)

La matriz de correlación del Grupo D muestra que:

- Las variables `weekday_is_*` están naturalmente relacionadas de forma negativa entre sí (si es lunes no puede ser martes, etc.), pero la correlación numérica se mantiene moderada gracias a la codificación one-hot.
- `is_weekend` está fuertemente ligada a `weekday_is_saturday` y `weekday_is_sunday`, lo cual es esperable porque agrupa precisamente esos dos días.

Aunque no hay una multicolinealidad extrema como en variables numéricas, sí es importante considerar que `is_weekend` aporta información redundante respecto a las dummies de sábado y domingo. Dependiendo del modelo, podría optarse por:
- Mantener todas las dummies.
- O bien, eliminar `is_weekend` si se busca una representación mínima sin redundancias.


🟪 11. Importancia de variables (modelo base Random Forest)

In [ ]:
# =====================================================================
# 11. Importancia de variables del Grupo D con RandomForest
# =====================================================================

from sklearn.ensemble import RandomForestRegressor

X_D = df[channel_cols + weekday_cols + [weekend_col]]
y = df["shares"]

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_D, y)

importances = pd.Series(rf.feature_importances_, index=X_D.columns)
importances_sorted = importances.sort_values(ascending=False)

print("📌 Importancia estimada de las variables (RandomForest - solo Grupo D)")
display(importances_sorted)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances_sorted.index, y=importances_sorted.values)
plt.title("Importancia de variables del Grupo D (RandomForest)")
plt.xticks(rotation=45)
plt.ylabel("Importancia")
plt.tight_layout()
plt.show()


### 11. Importancia de las variables del Grupo D

Al entrenar un modelo base de **Random Forest** únicamente con las variables del Grupo D, se observa que:

- Algunas variables de canal, como `data_channel_is_socmed` y `data_channel_is_lifestyle`, destacan con una importancia mayor, consistente con los promedios altos de *shares* observados.
- Entre los días, los relacionados con fin de semana (`weekday_is_saturday`, `weekday_is_sunday`, `is_weekend`) muestran una contribución relevante, lo que refuerza el patrón de mayor viralidad en esos días.
- Aunque la importancia individual de algunas dummies es baja, en conjunto, el grupo de canales y días aporta una señal clara y útil al modelo.

Este ejercicio sirve como una primera aproximación para justificar la inclusión de estas variables en el pipeline de modelado.


🟪 12. Relación “frecuencia vs viralidad” por canal y día

🔹 12.1 Frecuencia vs promedio de shares por canal

In [ ]:
# =====================================================================
# 12. Frecuencia vs viralidad (canales y días)
# =====================================================================

# ---- Canales ----
channel_stats_plot = pd.DataFrame({
    "frecuencia": df[channel_cols].sum(),
    "mean_shares": (
        df[channel_cols + ["shares"]]
        .melt(id_vars="shares", var_name="channel", value_name="is_channel")
        .query("is_channel == 1")
        .groupby("channel")["shares"]
        .mean()
    )
})

channel_stats_plot = channel_stats_plot.reset_index().rename(columns={"index": "channel"})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=channel_stats_plot,
                x="frecuencia", y="mean_shares",
                hue="channel")
plt.title("Frecuencia vs promedio de shares por canal")
plt.xlabel("Frecuencia (número de publicaciones)")
plt.ylabel("Promedio de shares")
plt.tight_layout()
plt.show()


🔹 12.2 Frecuencia vs promedio de shares por día

In [ ]:
# ---- Días de la semana ----
weekday_stats_plot = pd.DataFrame({
    "frecuencia": df[weekday_cols].sum(),
    "mean_shares": (
        df[weekday_cols + ["shares"]]
        .melt(id_vars="shares", var_name="weekday", value_name="is_day")
        .query("is_day == 1")
        .groupby("weekday")["shares"]
        .mean()
    )
})

weekday_stats_plot = weekday_stats_plot.reset_index().rename(columns={"index": "weekday"})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=weekday_stats_plot,
                x="frecuencia", y="mean_shares",
                hue="weekday")
plt.title("Frecuencia vs promedio de shares por día de la semana")
plt.xlabel("Frecuencia (número de publicaciones)")
plt.ylabel("Promedio de shares")
plt.tight_layout()
plt.show()


### 12. Frecuencia vs viralidad

Al comparar la **frecuencia de publicación** con el **promedio de shares**:

- Se confirma que los canales con mayor volumen (por ejemplo, World o Tech) **no son necesariamente** los más virales en promedio.
- Canales con menor frecuencia, como Social Media o Lifestyle, pueden tener **promedios de shares más altos**, lo que los convierte en nichos de alto impacto.
- En los días de la semana ocurre algo similar: los días con más publicaciones (martes, miércoles, jueves) no son los que alcanzan mayores niveles de viralidad; el fin de semana, aun con menor volumen, concentra más *shares* por noticia.

Este análisis es clave para entender que “publicar más” no garantiza “ser más viral”; lo que importa es la combinación **canal + día + contenido**.


🟪 13. Conclusiones ejecutivas del Grupo D

## Conclusiones ejecutivas — Grupo D (Canales y Días)

1. **Los canales no se comportan igual en términos de viralidad.**  
   Aunque canales como `World` y `Tech` concentran la mayor cantidad de publicaciones, otros como `Social Media` y `Lifestyle` logran promedios de *shares* más altos, lo que indica nichos de contenido especialmente virales.

2. **El día de publicación sí importa.**  
   Las métricas descriptivas y las pruebas ANOVA muestran diferencias significativas entre días: sábado y domingo presentan mayores promedios de *shares* y mayor dispersión, confirmando que el fin de semana es un periodo de alta viralidad.

3. **El fin de semana se comporta de forma distinta a los días laborales.**  
   El contraste de “entre semana vs fin de semana” mediante t-test confirma que publicar en fin de semana está asociado con un aumento estadísticamente significativo en los *shares*.

4. **La interacción Canal × Día explica mejor los picos de viralidad.**  
   Al analizar combinaciones específicas (canal–día), se detectan pares con impacto muy superior al promedio global, lo que sugiere que la estrategia óptima de publicación debe considerar ambos factores simultáneamente.

5. **Las variables del Grupo D son relevantes para el modelo.**  
   El análisis de importancia de variables mediante un modelo base de Random Forest muestra que dummies como `data_channel_is_socmed`, `data_channel_is_lifestyle`, `weekday_is_saturday`, `weekday_is_sunday` e `is_weekend` aportan una señal clara y útil para predecir *shares*.

En conjunto, el Grupo D aporta información estructural sobre **cuándo** y **por dónde** se publica el contenido, elementos clave para entender la dinámica de la viralidad en este dataset.


🟪 14. Implicaciones para el pipeline de MLOps

## Implicaciones para el pipeline de MLOps (Grupo D)

A partir del análisis del Grupo D, se derivan varias decisiones para el pipeline de modelado y operación:

1. **Selección de features.**  
   - Incluir las variables dummies de canales (`data_channel_is_*`) y días (`weekday_is_*`) en el conjunto de entrada del modelo.  
   - Mantener `is_weekend` como feature agregado, evaluando su impacto frente a mantener solo sábado y domingo (se puede comparar en experimentos de MLflow).

2. **Tratamiento de la distribución de shares.**  
   - Debido a la fuerte asimetría y presencia de outliers, es recomendable considerar una transformación (por ejemplo, `log10(shares + 1)`) como variable objetivo o como métrica de análisis complementaria.  
   - Este punto debe quedar documentado en los experimentos de entrenamiento.

3. **Monitoreo de drift en producción.**  
   - Implementar métricas de monitoreo sobre la distribución de:
     - Frecuencia de publicaciones por canal.
     - Frecuencia de publicaciones por día de la semana.
     - Proporción de publicaciones en fin de semana (`is_weekend`).
   - Cambios significativos en estos patrones pueden indicar **cambio de comportamiento de usuarios o estrategia editorial**, afectando la capacidad predictiva del modelo.

4. **Estrategia de experimentos.**  
   - Comparar modelos:
     - Con y sin `is_weekend`.
     - Agrupando días (laboral vs fin de semana) vs usando todos los días por separado.  
   - Registrar en MLflow qué configuración de codificación categórica ofrece mejor desempeño.

5. **Uso de insights para el negocio.**  
   Además del modelo, los resultados de este EDA pueden alimentar decisiones de negocio:
   - Definir horarios y días recomendados de publicación por canal.
   - Ajustar estrategia de contenido hacia los canales con mejor relación impacto/volumen.

Con esto, el análisis del Grupo D queda no solo descriptivo, sino conectado directamente con las decisiones de **feature engineering, monitoreo y experimentación** dentro del ciclo de MLOps.
